# Exercise 1: Segmenting words

*The Computational Phenotype · version 0.1.0*

> Segment *evlerimizden* and explain the contribution of each piece. Then compare *yazıldılar* and *yazılamadılar*. Use the analyses *yaz-ıl-dı-lar* and *yaz-ıl-a-ma-dı-lar* to identify passive voice, past and agreement, and explain how the second form adds negative ability. Under this chapter's surface-segmentation convention, how many pieces does each word contain? Why need this count not equal the number of language-model tokens?

**Setup.** `pip install "turkicnlp[hfst]==0.5.0" pandas tiktoken`. Download the language resources once with `turkicnlp.download(lang, processors=[...])`, as shown in the first code cell. Cells that fetch data from the web need internet access.

In [ ]:
import hfst
import turkicnlp
from turkicnlp.resources.registry import ModelRegistry

turkicnlp.download("tur", processors=["morph"])          # Apertium Turkish transducers (once)
tur_dir = ModelRegistry.get_model_path("tur", "morph", "apertium", script="Latn")

def load(path):
    stream = hfst.HfstInputStream(str(path)); fst = stream.read(); stream.close()
    fst.convert(hfst.ImplementationType.HFST_OLW_TYPE)   # optimized lookup: no epsilon symbols in output
    return fst

analyser = load(tur_dir / "tur.automorf.hfst")
generator = load(tur_dir / "tur.autogen.hfst")
analyse = lambda word: sorted({a for a, _ in analyser.lookup(word)})
generate = lambda analysis: sorted({f for f, _ in generator.lookup(analysis)})

In [ ]:
import pandas as pd

segmentations = {                                   # the chapter's surface-segmentation convention
    "evlerimizden":  [("ev", "house", "stem"), ("ler", "PL", "plural"), ("imiz", "POSS.1PL", "our"), ("den", "ABL", "from")],
    "yazıldılar":    [("yaz", "write", "stem"), ("ıl", "PASS", "passive"), ("dı", "PST", "past"), ("lar", "3PL", "they")],
    "yazılamadılar": [("yaz", "write", "stem"), ("ıl", "PASS", "passive"), ("a", "ABIL", "ability"),
                      ("ma", "NEG", "negation (with ABIL: inability)"), ("dı", "PST", "past"), ("lar", "3PL", "they")],
}
for word, pieces in segmentations.items():
    assert "".join(p for p, _, _ in pieces) == word          # the pieces rebuild the word
    print(f"{word}: {'-'.join(p for p, _, _ in pieces)}  ({len(pieces)} pieces)")
    print("   Apertium:", analyse(word)[0])

In [ ]:
# Language-model tokens for the same words (OpenAI's o200k_base tokenizer; downloads its vocabulary once)
try:
    import tiktoken
    enc = tiktoken.get_encoding("o200k_base")
    for word in segmentations:
        pieces = [enc.decode([t]) for t in enc.encode(word)]
        print(f"{word}: {len(pieces)} tokens {pieces}")
except Exception as exc:
    print("Tokenizer not available here:", type(exc).__name__, "- install tiktoken and allow its one-time download.")

## Worked solution

**evlerimizden** = *ev-ler-imiz-den* 'from our houses': stem *ev* 'house', plural *-ler*, first-person plural possessive *-imiz* 'our', ablative *-den* 'from'. Four pieces.

**yazıldılar** = *yaz-ıl-dı-lar* 'they were written': stem, passive *-ıl*, past *-dı*, third-person plural *-lar*. Four pieces.

**yazılamadılar** = *yaz-ıl-a-ma-dı-lar* 'they could not be written'. It inserts *-a-ma-* between the passive and the past. Under the chapter's convention *-a-* (ability) and *-ma-* (negation) are two pieces that together form the inability construction, so the word has **six** pieces. Negative ability is not positive ability (*-abil-*) plus negation: *yazılabildiler* 'they could be written' uses a different suffix. The Apertium analysis shows the same structure with tags: `yaz<v><tv><pass><abil><neg><ifi><p3><pl>`.

**Why tokens differ.** A subword tokeniser splits strings by frequency in its training data, not by morpheme boundaries. Its pieces can cross morpheme boundaries or merge several morphemes, and the number depends on the tokeniser, not on the grammar. For Turkish, the chapter's source measures 1.95–2.33 tokens per word on average across four LLM tokenisers. The token cell above shows the actual split for each word; compare it with the segmentation.